## Definir agente

In [ ]:
# 1. Importar librerías necesarias
from deepagents import create_deep_agent
from langchain_google_genai import ChatGoogleGenerativeAI # ¡Librería correcta para Gemini!
from dotenv import load_dotenv
import os

# 2. Cargar token desde el archivo .env
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

# Inicializar el modelo nativo de Google
llm = ChatGoogleGenerativeAI(
    google_api_key=api_key,
    model="gemini-1.5-flash", 
    temperature=0
)

# 3. Importar las Tools
from tools import cargar_bases_y_shapefile, semaforizar_distrito, calcular_mamografo_cercano

# 4. >>> SYSTEM PROMPT <<<
system_prompt = """You are an expert Geospatial Public Health Analyst for the Peruvian Government. 
Your goal is to evaluate the coverage of mammography services across Peruvian districts.

When asked about a district, load and follow the 'peru-mammography-analysis' skill.
Use the demographic data (women 40-69 years old) and the installed capacity (RENIPRESS categories) to determine the traffic-light status (Semaforización) of the district:
- VERDE: The installed capacity fully covers the local population demand.
- ÁMBAR: There is a mammograph, but the population demand exceeds the hospital's capacity.
- ROJO: No mammographs in the district. You MUST calculate the distance in kilometers to the closest available mammograph.
Report the exact statistics and provide output map paths when requested."""

# 5. Crear el GeoAgent conectando todo
agent = create_deep_agent(
    model=llm,
    tools=[cargar_bases_y_shapefile, semaforizar_distrito, calcular_mamografo_cercano],
    system_prompt=system_prompt,
    skills=["."]  # Esto le dice al agente que busque el archivo SKILL.md en la misma carpeta
)

## Probar agente

In [ ]:
# 1. Definir la consulta en lenguaje natural para el servidor público
query = """
Por favor, evalúa la cobertura de mamógrafos en el distrito de CHALHUANCA. 
Dime cuál es el estado del semáforo cruzando la demanda de afiliadas con la oferta. 
Si está en rojo, indícame a qué distancia en kilómetros se encuentra el establecimiento con mamógrafo más cercano.
"""

# 2. Ejecutar el agente (invoke)
print("Pensando y ejecutando tools... (esto puede tomar unos segundos)")
result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

# 3. Imprimir el resultado de manera legible usando 'rich'
from rich import print as rprint
rprint(result)